In [ ]:
from modular.URDF_writer import UrdfWriter

In [ ]:
urdf_writer = UrdfWriter(speedup=True, floating_base=True)
homing_joint_map = {}

In [ ]:
urdf_writer.add_module('concert/mobile_platform_concert.json', module_name='mobile_base')

In [ ]:
# Attach each steering and wheel pair to its corner of the mobile base.
steering_by_connector = (
    'concert/module_steering_concert_fl_rr.json',
    'concert/module_steering_concert_fr_rl.json',
    'concert/module_steering_concert_fr_rl.json',
    'concert/module_steering_concert_fl_rr.json',
)

for port, steering_filename in enumerate(steering_by_connector, start=1):
    urdf_writer.select_module_from_name(f'mobile_base_con{port}')
    wheel_data, steering_data = urdf_writer.add_wheel_module(
        wheel_filename='concert/module_wheel_concert.json',
        steering_filename=steering_filename,
    )
    homing_joint_map[steering_data['name']] = 0.0
    homing_joint_map[wheel_data['name']] = 0.0

In [ ]:
# Attach the manipulator to the fifth connector.
urdf_writer.select_module_from_name('mobile_base_con5')

# J1
data = urdf_writer.add_module('concert/module_joint_elbow_A_concert.json')
homing_joint_map[data['name']] = 0.5

# Add a 300 mm passive link.
urdf_writer.add_module('concert/module_link_straight_100_concert.json')

# J2
data = urdf_writer.add_module('concert/module_joint_elbow_A_concert.json')
homing_joint_map[data['name']] = -0.5

# Add a 400 mm passive link.
urdf_writer.add_module('concert/module_link_straight_200_concert.json')

# J3
data = urdf_writer.add_module('concert/module_joint_yaw_B_concert.json')
homing_joint_map[data['name']] = 0.0

# J4
data = urdf_writer.add_module('concert/module_joint_elbow_B_concert.json')
homing_joint_map[data['name']] = 0.5

# J5
data = urdf_writer.add_module('concert/module_joint_elbow_B_concert.json')
homing_joint_map[data['name']] = -0.5

# Simple end effector
urdf_writer.add_simple_ee(z_offset=0.2)

In [ ]:
urdf_writer.remove_all_connectors()
urdf_writer.write_urdf()
urdf_writer.write_lowlevel_config()
urdf_writer.write_problem_description_multi()
urdf_writer.write_srdf(builder_joint_map=homing_joint_map)
urdf_writer.write_joint_map()

In [ ]:
urdf_writer.deploy_robot("concert_robot")